# Prosit 2a + 2b Unified Pipeline
Combine vanilla CTC RNN baselines with attention-augmented variants in one configurable notebook. Each section keeps the educational flow while making it easy to swap model backbones, training hyperparameters, and visualisations.


## Roadmap
1. Environment setup & shared imports
2. Configurable data and experiment registry
3. Text preprocessing and feature extraction
4. Data preparation & loaders
5. Model zoo wired to `src/models`
6. Training, evaluation, and inference utilities
7. Interactive Plotly dashboards for metrics and attention maps


In [ ]:
# Optional: uncomment when running in a fresh environment (e.g., Colab)
!pip -q install torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install datasets==3.6.0 jiwer librosa accelerate rich plotly


In [ ]:
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple
import json
import math
import os
import random
import re
import sys
import unicodedata

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchaudio
from datasets import Audio, Dataset, load_dataset, load_dataset_builder
from jiwer import cer, wer
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Using Apple MPS")
else:
    DEVICE = "cpu"
    print("Using CPU")


Using CUDA GPU: NVIDIA A100-SXM4-40GB


## 1. Experiment Configuration
We wrap dataset, model, and optimisation knobs inside dataclasses so you can register multiple experiments (e.g., vanilla RNN, GRU, LSTM, Attention) and evaluate them in sequence.


In [ ]:
from __future__ import annotations
from collections import Counter
from dataclasses import dataclass
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple, Literal
import hashlib
import itertools
import json
import math
import os
import random
import re
import shutil
import sys
import unicodedata

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchaudio
from datasets import Audio, Dataset, DownloadConfig, DownloadMode, load_dataset, load_dataset_builder, load_from_disk
from jiwer import cer, wer
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd()

# Prefer VS Code's renderer but fall back gracefully when unavailable
for renderer in ("vscode", "notebook_connected", "browser"):
    try:
        pio.renderers.default = renderer
        break
    except ValueError:
        continue

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Using Apple MPS")
else:
    DEVICE = "cpu"
    print("Using CPU")

FEATURE_TYPES = ("mel", "mfcc")  # Supported acoustic feature representations

SAMPLE_PREVIEW_LIMIT = 10
SAMPLE_SAVE_LIMIT = 50
FINAL_METRICS_FILENAME = "final_metrics.json"
SAMPLE_PREDICTIONS_FILENAME = "sample_predictions.json"
TRAINING_HISTORY_FILENAME = "history.json"

HISTORY_CACHE: Dict[str, List[Dict[str, float]]] = {}
TEXT_TRANSFORM_CACHE: Dict[str, "TextTransform"] = {}


@dataclass
class DataConfig:
    dataset_id: str = "intronhealth/afrispeech-200"  # Hugging Face dataset to download/stream; swap when targeting another corpus
    dataset_config: str = "twi"  # Specific language/config split inside the dataset; adjust for dialect or language changes
    sample_rate: int = 16_000  # Target sampling rate (Hz); resample audio here if the source differs
    feature_type: Literal["mel", "mfcc"] = "mel"  # Choose between mel-spectrogram or MFCC features
    n_mels: int = 80  # Number of Mel bins in the spectrogram; raise for richer features, lower for speed
    n_mfcc: int = 13  # Number of MFCCs to extract (if feature_type is MFCC)
    frame_length_ms: int = 25  # STFT window size in milliseconds; tweak to capture broader/narrower temporal context
    frame_shift_ms: int = 10  # Hop length between frames (ms); smaller hops yield finer time resolution
    max_seconds: Optional[float] = 30.0  # Drop utterances longer than this duration; increase when training on full-length speech
    fraction: int = 50  # Percentage of the dataset to sample; ramp up toward 100 for full training
    streaming_buffer: int = 10_000  # Buffer size used during streaming shuffle; raise if running into shuffle bias
    seed: int = SEED  # Seed for deterministic sampling and shuffling; change to vary random subsets

    def fingerprint(self) -> str:
        # Canonical JSON string used as a cache key for dataset and preprocessing reuse
        payload = {
            "dataset_config": self.dataset_config,
            "dataset_id": self.dataset_id,
            "fraction": self.fraction,
            "frame_length_ms": self.frame_length_ms,
            "frame_shift_ms": self.frame_shift_ms,
            "max_seconds": self.max_seconds,
            "feature_type": self.feature_type,
            "n_mels": self.n_mels,
            "n_mfcc": self.n_mfcc,
            "sample_rate": self.sample_rate,
            "seed": self.seed,
            "streaming_buffer": self.streaming_buffer,
        }
        return json.dumps(payload, sort_keys=True, default=str)



@dataclass
class ModelConfig:
    name: str  # Human-readable label shown in logs/plots
    arch: str  # Backbone identifier ("rnn", "gru", "lstm", "attention"); selects the sequence model
    hidden_size: int = 256  # Width of recurrent/attention layers; increase for capacity, decrease for memory savings
    num_layers: int = 3  # Depth of stacked recurrent blocks; more layers typically improve modeling power
    bidirectional: bool = True  # Enable BLSTM/BiGRU behaviour; disable for streaming or causal setups
    attn_heads: int = 4  # Number of heads in the self-attention block (used when arch == "attention")
    attn_dropout: float = 0.1  # Dropout applied inside attention; raise to regularise, lower if underfitting
    dropout: float = 0.2  # Generic dropout for recurrent outputs/classifier; tune based on over/underfitting



@dataclass
class TrainConfig:
    batch_size: int = 8  # Mini-batch size; limited by GPU/TPU memory—raise cautiously for stability
    epochs: int = 50  # Number of training passes over the dataset; extend for better convergence
    learning_rate: float = 3e-4  # Base LR for AdamW/OneCycle; tune alongside scheduler behaviour
    weight_decay: float = 1e-5  # L2 regularisation term; reduce if gradients vanish, increase to curb overfitting
    grad_clip: float = 5.0  # Gradient norm clipping threshold to stabilise training
    apply_specaugment: bool = True  # Enable SpecAugment masks; disable for debugging clean inputs
    use_onecycle: bool = True  # Toggle OneCycleLR; set False to use a constant LR schedule
    save_dir: Path = PROJECT_ROOT / "artifacts_prosit2"  # Root folder for checkpoints and history files



@dataclass
class ExperimentConfig:
    key: str  # Unique identifier used in caches, file names, and dashboards
    data: DataConfig  # Data-related hyperparameters and caching controls
    model: ModelConfig  # Architecture hyperparameters for this experiment
    train: TrainConfig  # Training loop configuration tied to this experiment

    def artifact_dir(self, create: bool = True) -> Path:
        # Compute the filesystem path where this experiment stores artifacts
        path = self.train.save_dir / self.key
        if create:
            path.mkdir(parents=True, exist_ok=True)  # Ensure the directory exists before writing checkpoints
        return path

Using CUDA GPU: NVIDIA A100-SXM4-40GB


## 2. Data Access Helpers
We stream AfriSpeech-200 (Akan) with deterministic sampling so each experiment sees the same subset. Adjust `DataConfig.fraction` and `max_seconds` when you want a larger or smaller curriculum.

`describe_data_cache()` summarises whether a download already lives on disk, while `configure_data_cache(...)` lets you toggle skip/force behaviours for Colab sweeps. Call `clear_data_cache("downloads")` or `clear_data_cache("memory")` whenever you need a clean slate without editing the training code.

In [ ]:
from itertools import islice


DATASET_CACHE: Dict[str, Tuple[Dataset, Dataset]] = {}
PROCESSED_DATASET_CACHE: Dict[str, Tuple[Dataset, Dataset]] = {}


COLAB_ROOT = Path("/content")
DEFAULT_CACHE_ROOT = (COLAB_ROOT / "prosit2_cache") if COLAB_ROOT.exists() else (PROJECT_ROOT / "prosit2_cache")

DATA_RUNTIME_OPTIONS: Dict[str, Any] = {
    "cache_root": DEFAULT_CACHE_ROOT,
    "skip_download": True,
    "force_download": False,
    "force_reprocess": False,
}


def _ensure_cache_root(path: Optional[Path] = None) -> Path:
    root = Path(path if path is not None else DATA_RUNTIME_OPTIONS["cache_root"]).expanduser().resolve()
    root.mkdir(parents=True, exist_ok=True)
    return root


def describe_data_cache() -> Dict[str, Any]:
    cache_root = _ensure_cache_root()
    summary = {
        "cache_root": str(cache_root),
        "skip_download": DATA_RUNTIME_OPTIONS["skip_download"],
        "force_download": DATA_RUNTIME_OPTIONS["force_download"],
        "force_reprocess": DATA_RUNTIME_OPTIONS["force_reprocess"],
        "has_cached_dataset": _has_cached_dataset(cache_root),
    }
    print(json.dumps(summary, indent=2))
    return summary


def configure_data_cache(
    cache_root: Optional[str] = None,
    skip_download: Optional[bool] = None,
    force_download: Optional[bool] = None,
    force_reprocess: Optional[bool] = None,
    ) -> Dict[str, Any]:
    if cache_root is not None:
        DATA_RUNTIME_OPTIONS["cache_root"] = Path(cache_root).expanduser().resolve()
    if skip_download is not None:
        DATA_RUNTIME_OPTIONS["skip_download"] = bool(skip_download)
    if force_download is not None:
        DATA_RUNTIME_OPTIONS["force_download"] = bool(force_download)
    if force_reprocess is not None:
        DATA_RUNTIME_OPTIONS["force_reprocess"] = bool(force_reprocess)
    _ensure_cache_root()
    return dict(DATA_RUNTIME_OPTIONS)


@contextmanager
def data_runtime_context(**overrides: Any):
    previous = dict(DATA_RUNTIME_OPTIONS)
    try:
        if overrides:
            configure_data_cache(**overrides)
        yield dict(DATA_RUNTIME_OPTIONS)
    finally:
        DATA_RUNTIME_OPTIONS.clear()
        DATA_RUNTIME_OPTIONS.update(previous)
        _ensure_cache_root(previous.get("cache_root"))


def clear_data_cache(scope: str = "all") -> None:
    scope_normalized = scope.lower()
    cleared_any = False
    if scope_normalized in {"all", "memory"}:
        DATASET_CACHE.clear()
        PROCESSED_DATASET_CACHE.clear()
        TEXT_TRANSFORM_CACHE.clear()
        cleared_any = True
        print("Cleared in-memory dataset caches.")
    if scope_normalized in {"all", "downloads", "raw"}:
        cache_root = Path(DATA_RUNTIME_OPTIONS["cache_root"]).expanduser().resolve()
        if cache_root.exists():
            shutil.rmtree(cache_root)
            cleared_any = True
            print(f"Removed cached downloads in {cache_root}.")
    if not cleared_any:
        print(f"No matching cache scope found for '{scope}'.")
    _ensure_cache_root()


def _has_cached_dataset(cache_root: Path) -> bool:
    datasets_dir = cache_root / "datasets"
    downloads_dir = cache_root / "downloads"
    return (datasets_dir.exists() and any(datasets_dir.iterdir())) or (downloads_dir.exists() and any(downloads_dir.iterdir()))


def _dataset_cache_key(data_cfg: DataConfig) -> str:
    return data_cfg.fingerprint()



def _calc_limit(total: Optional[int], fraction: int) -> Optional[int]:
    if total is None or fraction >= 100:
        return total
    return max(1, int(total * fraction / 100))



def _stream_subset(stream, limit: Optional[int], data_cfg: DataConfig):
    shuffled = stream.shuffle(seed=data_cfg.seed, buffer_size=data_cfg.streaming_buffer)
    if data_cfg.max_seconds is not None:
        shuffled = shuffled.filter(lambda x: float(x.get("duration", 1e9)) <= data_cfg.max_seconds)
    if limit is None:
        return list(shuffled)
    return list(islice(shuffled, limit))



def _download_config() -> DownloadConfig:
    cache_root = _ensure_cache_root()
    return DownloadConfig(cache_dir=str(cache_root))



def _processed_cache_paths(exp: ExperimentConfig, text_transform: TextTransform) -> Tuple[Path, Path]:
    cache_root = _ensure_cache_root()
    processed_root = cache_root / "processed"
    processed_root.mkdir(parents=True, exist_ok=True)
    payload = {
        "data": exp.data.fingerprint(),
        "vocab": len(text_transform.tokens),
        "feature": exp.data.feature_type,
        "n_mels": exp.data.n_mels,
        "n_mfcc": exp.data.n_mfcc,
    }
    digest = hashlib.sha1(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()[:12]
    prefix = f"{exp.key}-{digest}"
    return processed_root / f"{prefix}-train", processed_root / f"{prefix}-test"



def ensure_dataset_ready(data_cfg: DataConfig, *, verbose: bool = True) -> Path:
    runtime = DATA_RUNTIME_OPTIONS
    cache_root = _ensure_cache_root(runtime.get("cache_root"))
    skip_download = bool(runtime.get("skip_download", True))
    force_download = bool(runtime.get("force_download", False))
    has_cache = _has_cached_dataset(cache_root)
    download_mode = DownloadMode.FORCE_REDOWNLOAD if force_download else DownloadMode.REUSE_DATASET_IF_EXISTS

    if skip_download and has_cache and not force_download:
        if verbose:
            print(f"Using cached download in {cache_root}")
    else:
        if verbose:
            action = "Forcing fresh download" if force_download else "Preparing dataset cache"
            print(f"{action} for {data_cfg.dataset_id} ({data_cfg.dataset_config}) in {cache_root}")
        builder = load_dataset_builder(data_cfg.dataset_id, data_cfg.dataset_config, trust_remote_code=True)
        builder.download_and_prepare(
            download_mode=download_mode,
            download_config=_download_config(),
        )
        if force_download:
            DATA_RUNTIME_OPTIONS["force_download"] = False
            DATASET_CACHE.clear()
            PROCESSED_DATASET_CACHE.clear()
            TEXT_TRANSFORM_CACHE.clear()

    return cache_root



_ensure_cache_root()  # Prepare the default cache folder on import



def load_afrispeech_dataset(data_cfg: DataConfig) -> Tuple[Dataset, Dataset]:
    cache_key = _dataset_cache_key(data_cfg)
    if cache_key in DATASET_CACHE:
        return DATASET_CACHE[cache_key]

    ensure_dataset_ready(data_cfg)
    builder = load_dataset_builder(data_cfg.dataset_id, data_cfg.dataset_config, trust_remote_code=True)
    splits = getattr(builder.info, "splits", {}) or {}
    total_train = splits.get("train").num_examples if "train" in splits else None
    total_test = splits.get("test").num_examples if "test" in splits else None

    train_stream = load_dataset(
        data_cfg.dataset_id,
        data_cfg.dataset_config,
        split="train",
        streaming=True,
        trust_remote_code=True,
        download_config=_download_config(),
    )
    test_stream = load_dataset(
        data_cfg.dataset_id,
        data_cfg.dataset_config,
        split="test",
        streaming=True,
        trust_remote_code=True,
        download_config=_download_config(),
    )

    train_limit = _calc_limit(total_train, data_cfg.fraction)
    test_limit = _calc_limit(total_test, data_cfg.fraction)

    train_records = _stream_subset(train_stream, train_limit, data_cfg)
    test_records = _stream_subset(test_stream, test_limit, data_cfg)

    train_ds = Dataset.from_list(train_records).cast_column("audio", Audio(sampling_rate=data_cfg.sample_rate))
    test_ds = Dataset.from_list(test_records).cast_column("audio", Audio(sampling_rate=data_cfg.sample_rate))

    DATASET_CACHE[cache_key] = (train_ds, test_ds)
    return train_ds, test_ds


# Quick reference: uncomment the calls you need in a Colab runtime
## describe_data_cache()

## configure_data_cache(skip_download=False)   # Download immediately even if cache exists
## configure_data_cache(force_download=True)  # Force Hugging Face to re-download on the next call
## configure_data_cache(force_reprocess=True) # Regenerate processed feature datasets
## clear_data_cache("downloads")             # Remove on-disk cache under the current cache_root
## clear_data_cache("memory")                # Drop only the in-memory Dataset objects

In [ ]:
EXPERIMENTS: Dict[str, ExperimentConfig] = {
    "vanilla_rnn": ExperimentConfig(
        key="vanilla_rnn",
        data=DataConfig(feature_type="mel", fraction=50),
        model=ModelConfig(name="Vanilla BiRNN", arch="rnn", hidden_size=256, num_layers=3, bidirectional=True, dropout=0.2),
        train=TrainConfig(epochs=20, batch_size=8, learning_rate=3e-4, use_onecycle=True),
    ),
    "gru": ExperimentConfig(
        key="gru",
        data=DataConfig(feature_type="mel", fraction=60),
        model=ModelConfig(name="BiGRU", arch="gru", hidden_size=320, num_layers=3, bidirectional=True, dropout=0.25),
        train=TrainConfig(epochs=20, batch_size=8, learning_rate=2.5e-4, use_onecycle=True),
    ),
    "lstm": ExperimentConfig(
        key="lstm",
        data=DataConfig(feature_type="mfcc", fraction=60, n_mfcc=20, n_mels=128),
        model=ModelConfig(name="BiLSTM", arch="lstm", hidden_size=384, num_layers=3, bidirectional=True, dropout=0.3),
        train=TrainConfig(epochs=20, batch_size=8, learning_rate=2e-4, use_onecycle=True),
    ),
    "attention": ExperimentConfig(
        key="attention",
        data=DataConfig(feature_type="mel", fraction=50, n_mels=100),
        model=ModelConfig(name="Self-Attention RNN", arch="attention", hidden_size=256, num_layers=2, bidirectional=True, attn_heads=4, dropout=0.2),
        train=TrainConfig(epochs=20, batch_size=6, learning_rate=2e-4, use_onecycle=False),
    ),
}

print(f"Registered experiments: {', '.join(EXPERIMENTS.keys())}")

Registered experiments: vanilla_rnn, gru, lstm, attention


## 3. Text Normalisation & Vocabulary
We reuse the unicode-aware cleaners from Prosit 2a, wrap them in a `TextTransform`, and share across experiments so decoding stays consistent.


In [ ]:
TEXT_CLEAN_PATTERN = re.compile(r"[^a-z' ]+")

def normalize_text(text: str) -> str:
    if not text:
        return ""
    normalized = unicodedata.normalize("NFKD", text)
    stripped = "".join(ch for ch in normalized if not unicodedata.combining(ch))
    lowered = stripped.lower().replace("’", "'").replace("`", "'")
    cleaned = TEXT_CLEAN_PATTERN.sub(" ", lowered)
    compact = re.sub(r"\s+", " ", cleaned).strip()
    return compact


class TextTransform:
    """Character-level tokenizer with CTC-friendly specials."""
    def __init__(self, tokens: Iterable[str]):
        unique_chars = sorted(set(tokens))
        specials = ["<blank>", "<space>", "<unk>"]
        ordered = specials + [ch for ch in unique_chars if ch not in specials]
        self.tokens: List[str] = ordered
        self.token_to_id: Dict[str, int] = {tok: idx for idx, tok in enumerate(ordered)}
        self.id_to_token: Dict[int, str] = {idx: tok for tok, idx in self.token_to_id.items()}
        self.blank_id: int = self.token_to_id["<blank>"]
        self.space_id: int = self.token_to_id["<space>"]
        self.unk_id: int = self.token_to_id["<unk>"]

    def text_to_ids(self, text: str) -> List[int]:
        sequence: List[int] = []
        for char in text:
            if char == " ":
                sequence.append(self.space_id)
            else:
                sequence.append(self.token_to_id.get(char, self.unk_id))
        return sequence

    def ids_to_text(self, ids: Iterable[int]) -> str:
        characters: List[str] = []
        for idx in ids:
            token = self.id_to_token.get(int(idx), "<unk>")
            if token == "<space>":
                characters.append(" ")
            elif token in {"<blank>", "<unk>"}:
                continue
            else:
                characters.append(token)
        return "".join(characters).strip()


def _collect_charset(dataset: Dataset) -> Counter:
    counts: Counter = Counter()
    for example in dataset:
        normalized = normalize_text(example.get("transcript", ""))
        counts.update(char for char in normalized if char and char != " ")
    return counts


def get_text_transform(data_cfg: DataConfig, dataset: Dataset, min_frequency: int = 1) -> TextTransform:
    cache_key = f"{data_cfg.fingerprint()}::vocab::{len(dataset)}::min{min_frequency}"
    cached = TEXT_TRANSFORM_CACHE.get(cache_key)
    if cached is not None:
        return cached
    charset_counts = _collect_charset(dataset)
    tokens = [char for char, freq in charset_counts.items() if freq >= min_frequency]
    transform = TextTransform(tokens)
    TEXT_TRANSFORM_CACHE[cache_key] = transform
    return transform

## 4. Acoustic Features
Log-Mel spectrograms with per-utterance normalisation mirror Prosit 2a. These features feed every backbone uniformly.


In [ ]:
from typing import TYPE_CHECKING
import torch.nn as nn
import torch.nn.functional as F

from torch.nn.utils.rnn import pad_sequence

if TYPE_CHECKING:  # Prevent circular imports during type checking
    from .b37a6bf8 import DataConfig  # Import DataConfig for type hinting

def build_feature_extractor(data_cfg: 'DataConfig'):  # Use string literal for type hinting
    n_fft = int(data_cfg.sample_rate * data_cfg.frame_length_ms / 1000)
    hop_length = int(data_cfg.sample_rate * data_cfg.frame_shift_ms / 1000)

    def extract(waveform: np.ndarray, sample_rate: int) -> torch.Tensor:
        tensor = torch.from_numpy(waveform).float()
        if sample_rate != data_cfg.sample_rate:
            tensor = torchaudio.functional.resample(tensor, sample_rate, data_cfg.sample_rate)
        if tensor.dim() == 1:
            tensor = tensor.unsqueeze(0)

        if data_cfg.feature_type == "mel":  # Compare against plain string
            mel_transform = torchaudio.transforms.MelSpectrogram(
                sample_rate=data_cfg.sample_rate,
                n_fft=n_fft,
                hop_length=hop_length,
                n_mels=data_cfg.n_mels,
                center=True,
                power=2.0,
            )
            to_db = torchaudio.transforms.AmplitudeToDB()
            spec = to_db(mel_transform(tensor)).squeeze(0).transpose(0, 1)

        elif data_cfg.feature_type == "mfcc":  # Compare against plain string
            mfcc_transform = torchaudio.transforms.MFCC(
                sample_rate=data_cfg.sample_rate,
                n_mfcc=data_cfg.n_mfcc,
                melkwargs={
                    "n_fft": n_fft,
                    "hop_length": hop_length,
                    "n_mels": data_cfg.n_mels,  # Still use n_mels for the MelSpectrogram calculation within MFCC
                    "center": True,
                },
            )
            spec = mfcc_transform(tensor).squeeze(0).transpose(0, 1)

        else:
            raise ValueError(f"Unknown feature type: {data_cfg.feature_type}")

        # Apply per-utterance normalization
        mean = spec.mean(dim=0, keepdim=True)
        std = spec.std(dim=0, keepdim=True) + 1e-5
        return (spec - mean) / std

    return extract

## 5. Dataset Preparation
We attach features, labels, and bookkeeping fields (lengths, transcripts) to each example so loaders work identically across runs.


In [ ]:
def prepare_dataset(dataset: Dataset, extractor, text_transform: TextTransform) -> Dataset:
    keep_cols = ["audio", "transcript"]

    def _map(example):
        waveform = example["audio"]["array"]
        sr = example["audio"]["sampling_rate"]
        normalized_text = normalize_text(example["transcript"])
        features = extractor(waveform, sr)
        features_np = features.detach().cpu().numpy().astype(np.float32)
        labels = text_transform.text_to_ids(normalized_text)
        labels_array = np.asarray(labels, dtype=np.int64)

        example["normalized_transcript"] = normalized_text
        example["input_features"] = features_np.tolist()
        example["labels"] = labels_array.tolist()
        example["input_length"] = int(features_np.shape[0])
        example["target_length"] = int(labels_array.shape[0])
        return example

    processed = dataset.map(
        _map,
        remove_columns=[col for col in dataset.column_names if col not in keep_cols],
    )
    return processed


def ctc_collate(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    features = []
    labels = []
    for item in batch:
        feats = item["input_features"]
        if not isinstance(feats, torch.Tensor):
            feats = torch.tensor(feats, dtype=torch.float32)
        features.append(feats)

        lab = item["labels"]
        if not isinstance(lab, torch.Tensor):
            lab = torch.tensor(lab, dtype=torch.long)
        labels.append(lab)

    lengths = torch.tensor([feat.shape[0] for feat in features], dtype=torch.long)
    max_len = int(lengths.max())
    n_mels = features[0].shape[1]
    batch_size = len(features)

    padded = torch.zeros(max_len, batch_size, n_mels, dtype=torch.float32)
    for idx, feat in enumerate(features):
        padded[: feat.shape[0], idx] = feat

    target_lengths = torch.tensor([lab.shape[0] for lab in labels], dtype=torch.long)
    targets = torch.cat(labels) if labels else torch.tensor([], dtype=torch.long)

    return {
        "inputs": padded,
        "input_lengths": lengths,
        "targets": targets,
        "target_lengths": target_lengths,
    }


def build_dataloaders(train_ds: Dataset, test_ds: Dataset, train_cfg: TrainConfig) -> Tuple[DataLoader, DataLoader]:
    train_loader = DataLoader(train_ds, batch_size=train_cfg.batch_size, shuffle=True, collate_fn=ctc_collate)
    test_loader = DataLoader(test_ds, batch_size=train_cfg.batch_size, shuffle=False, collate_fn=ctc_collate)
    return train_loader, test_loader


def get_prepared_datasets(
    exp: ExperimentConfig,
    text_transform: TextTransform,
    raw_train: Optional[Dataset] = None,
    raw_test: Optional[Dataset] = None,
    ) -> Tuple[Dataset, Dataset]:
    runtime = DATA_RUNTIME_OPTIONS
    force_reprocess = bool(runtime.get("force_reprocess", False))
    cache_key = f"{exp.data.fingerprint()}::prep::{len(text_transform.tokens)}::{exp.data.n_mels}"
    if not force_reprocess and cache_key in PROCESSED_DATASET_CACHE:
        return PROCESSED_DATASET_CACHE[cache_key]

    train_cache_path, test_cache_path = _processed_cache_paths(exp, text_transform)

    if not force_reprocess and train_cache_path.exists() and test_cache_path.exists():
        proc_train = load_from_disk(str(train_cache_path))
        proc_test = load_from_disk(str(test_cache_path))
        PROCESSED_DATASET_CACHE[cache_key] = (proc_train, proc_test)
        print(f"Loaded processed features from {train_cache_path.parent}")
        return proc_train, proc_test

    if force_reprocess:
        PROCESSED_DATASET_CACHE.pop(cache_key, None)
        for path in (train_cache_path, test_cache_path):
            if path.exists():
                shutil.rmtree(path)
        print("Recomputing processed datasets from scratch.")

    if raw_train is None or raw_test is None:
        raw_train, raw_test = load_afrispeech_dataset(exp.data)
    extractor = build_feature_extractor(exp.data)
    proc_train = prepare_dataset(raw_train, extractor, text_transform)
    proc_test = prepare_dataset(raw_test, extractor, text_transform)

    train_cache_path.parent.mkdir(parents=True, exist_ok=True)
    proc_train.save_to_disk(str(train_cache_path))
    proc_test.save_to_disk(str(test_cache_path))
    if force_reprocess:
        DATA_RUNTIME_OPTIONS["force_reprocess"] = False
    PROCESSED_DATASET_CACHE[cache_key] = (proc_train, proc_test)
    print(f"Cached processed datasets under {train_cache_path.parent}")
    return proc_train, proc_test



## 6. Model Factory (Prosit 2a & 2b)
We tap directly into `Prosit 2/rnn-time-series-prediction/src/models` for backbone definitions and enrich them with CTC-friendly heads. Each variant shares a common interface so experiments differ only by configuration.


In [ ]:
class PrototypeRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.num_layers = num_layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
        out, _ = self.rnn(x, h0)
        return self.fc(out[:, -1, :])


class PrototypeGRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.output_size = output_size
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
        out, _ = self.gru(x, h0)
        return self.fc(out[:, -1, :])


class PrototypeLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.output_size = output_size
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
        out, _ = self.lstm(x, (h0, c0))
        return self.fc(out[:, -1, :])


class PrototypeAttention(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.num_layers = num_layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.attention_weights = None

    def forward(self, x):
        rnn_out, _ = self.rnn(x)
        attn = torch.softmax(rnn_out, dim=1)
        self.attention_weights = attn
        context = torch.bmm(attn.transpose(1, 2), rnn_out)
        return self.fc(context.squeeze(1))

    def get_attention_weights(self):
        return self.attention_weights

In [ ]:
# Inline replicas of Prosit 2 prototype models so the notebook works on hosted runtimes
class PrositAcousticModel(nn.Module):
    def __init__(
        self,
        arch: str,
        input_dim: int,
        hidden_size: int,
        num_layers: int,
        bidirectional: bool,
        vocab_size: int,
        dropout: float = 0.2,
        attn_heads: int = 4,
        attn_dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.arch = arch
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.prototype = None

        self.input_norm = nn.LayerNorm(input_dim)
        self.input_proj = nn.Linear(input_dim, hidden_size)

        rnn_kwargs = dict(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
            batch_first=False,
        )

        if arch == "rnn":
            self.prototype = PrototypeRNN(input_size=input_dim, hidden_size=hidden_size, output_size=hidden_size, num_layers=num_layers)
            self.sequence_model = nn.RNN(**rnn_kwargs, nonlinearity="tanh")
            self.attention = None
        elif arch == "gru":
            self.prototype = PrototypeGRU(input_size=input_dim, hidden_size=hidden_size, num_layers=num_layers, output_size=hidden_size)
            self.sequence_model = nn.GRU(**rnn_kwargs)
            self.attention = None
        elif arch == "lstm":
            self.prototype = PrototypeLSTM(input_size=input_dim, hidden_size=hidden_size, num_layers=num_layers, output_size=hidden_size)
            self.sequence_model = nn.LSTM(**rnn_kwargs)
            self.attention = None
        elif arch == "attention":
            self.prototype = PrototypeAttention(input_size=input_dim, hidden_size=hidden_size, output_size=hidden_size, num_layers=num_layers)
            self.sequence_model = nn.RNN(**rnn_kwargs, nonlinearity="tanh")
            self.attention = nn.MultiheadAttention(
                embed_dim=hidden_size * (2 if bidirectional else 1),
                num_heads=attn_heads,
                dropout=attn_dropout,
                batch_first=False,
            )
        else:
            raise ValueError(f"Unknown architecture: {arch}")

        out_dim = hidden_size * (2 if bidirectional else 1)
        self.output_norm = nn.LayerNorm(out_dim)
        self.classifier = nn.Sequential(
            nn.Linear(out_dim, out_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(out_dim // 2, vocab_size),
        )
        self.last_attention_map: Optional[torch.Tensor] = None

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(self.input_norm(inputs))
        outputs, _ = self.sequence_model(x)

        if self.attention is not None:
            attn_out, weights = self.attention(outputs, outputs, outputs, need_weights=True, average_attn_weights=False)
            outputs = outputs + attn_out
            self.last_attention_map = weights.detach().mean(dim=0)
        else:
            self.last_attention_map = None

        logits = self.classifier(self.output_norm(outputs))
        return F.log_softmax(logits, dim=-1)

    def greedy_decode(self, log_probs: torch.Tensor, input_lengths: torch.Tensor, blank_id: int, text_transform: TextTransform) -> List[str]:
        predictions = log_probs.argmax(dim=-1)
        transcripts: List[str] = []
        for batch_idx in range(predictions.shape[1]):
            limit = input_lengths[batch_idx].item()
            prev = None
            decoded = []
            for t in range(limit):
                token = predictions[t, batch_idx].item()
                if token != blank_id and token != prev:
                    decoded.append(token)
                prev = token
            transcripts.append(text_transform.ids_to_text(decoded))
        return transcripts

    def attention_map(self) -> Optional[torch.Tensor]:
        return self.last_attention_map


In [ ]:
def build_model(exp: ExperimentConfig, vocab_size: int) -> PrositAcousticModel:
    model_cfg = exp.model
    data_cfg = exp.data  # Get data config

    # Determine input dimension based on feature type
    if data_cfg.feature_type == "mel":
        input_dim = data_cfg.n_mels
    elif data_cfg.feature_type == "mfcc":
        input_dim = data_cfg.n_mfcc
    else:
        raise ValueError(f"Unknown feature type: {data_cfg.feature_type}")

    model = PrositAcousticModel(
        arch=model_cfg.arch,
        input_dim=input_dim,  # Use determined input_dim
        hidden_size=model_cfg.hidden_size,
        num_layers=model_cfg.num_layers,
        bidirectional=model_cfg.bidirectional,
        vocab_size=vocab_size,
        dropout=model_cfg.dropout,
        attn_heads=model_cfg.attn_heads,
        attn_dropout=model_cfg.attn_dropout,
    )
    return model.to(DEVICE)


def build_optimizer(exp: ExperimentConfig, model: nn.Module, steps_per_epoch: int) -> Tuple[torch.optim.Optimizer, Optional[torch.optim.lr_scheduler._LRScheduler]]:
    train_cfg = exp.train
    optimizer = torch.optim.AdamW(model.parameters(), lr=train_cfg.learning_rate, weight_decay=train_cfg.weight_decay)
    scheduler = None
    if train_cfg.use_onecycle:
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=train_cfg.learning_rate,
            epochs=train_cfg.epochs,
            steps_per_epoch=max(1, steps_per_epoch),
            pct_start=0.1,
            anneal_strategy="cos",
        )
    return optimizer, scheduler

## 7. Training & Evaluation Utilities
We keep the 2a loop (SpecAugment, CTC) and extend it with hooks for attention visualisation and experiment tracking.


In [ ]:
freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=8)
time_mask = torchaudio.transforms.TimeMasking(time_mask_param=20)
ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)


def maybe_augment(inputs: torch.Tensor, enabled: bool) -> torch.Tensor:
    if not enabled:
        return inputs
    augmented = inputs.permute(1, 2, 0)  # (batch, feature, time)
    augmented = freq_mask(augmented)
    augmented = time_mask(augmented)
    return augmented.permute(2, 0, 1)


def train_one_epoch(model: PrositAcousticModel, loader: DataLoader, exp: ExperimentConfig, optimizer, scheduler) -> float:
    model.train()
    train_cfg = exp.train
    running_loss = 0.0
    steps = 0
    for batch in loader:
        inputs = batch["inputs"].to(DEVICE)
        inputs = maybe_augment(inputs, enabled=train_cfg.apply_specaugment)
        input_lengths = batch["input_lengths"].to(DEVICE)
        targets = batch["targets"].to(DEVICE)
        target_lengths = batch["target_lengths"].to(DEVICE)

        optimizer.zero_grad()
        log_probs = model(inputs)

        if DEVICE == "mps":
            loss = ctc_loss(log_probs.cpu(), targets.cpu(), input_lengths.cpu(), target_lengths.cpu()).to(DEVICE)
        else:
            loss = ctc_loss(log_probs, targets, input_lengths, target_lengths)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), train_cfg.grad_clip)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        running_loss += loss.item()
        steps += 1
    return running_loss / max(steps, 1)


@torch.no_grad()
def evaluate_model(
    model: PrositAcousticModel,
    loader: DataLoader,
    text_transform: TextTransform,
    max_batches: Optional[int] = None,
    sample_limit: int = SAMPLE_PREVIEW_LIMIT,
    ) -> Dict[str, Any]:
    model.eval()
    references: List[str] = []
    hypotheses: List[str] = []
    sample_records: List[Dict[str, str]] = []
    batches = 0
    effective_limit = max(0, min(sample_limit, SAMPLE_SAVE_LIMIT))
    for batch in loader:
        inputs = batch["inputs"].to(DEVICE)
        input_lengths = batch["input_lengths"].to(DEVICE)
        targets = batch["targets"].cpu()
        target_lengths = batch["target_lengths"].cpu()

        log_probs = model(inputs)
        decoded = model.greedy_decode(log_probs.cpu(), input_lengths.cpu(), blank_id=0, text_transform=text_transform)

        batch_references: List[str] = []
        start = 0
        for length in target_lengths:
            ids = targets[start : start + length]
            start += length
            transcript = text_transform.ids_to_text(ids.tolist())
            batch_references.append(transcript)
            references.append(transcript)
        hypotheses.extend(decoded)

        if effective_limit > 0 and len(sample_records) < effective_limit:
            remaining = effective_limit - len(sample_records)
            for ref, hyp in itertools.islice(zip(batch_references, decoded), remaining):
                sample_records.append({
                    "reference": ref,
                    "prediction": hyp,
                })

        batches += 1
        if max_batches is not None and batches >= max_batches:
            break

    metrics = {
        "wer": float(wer(references, hypotheses)) if references else 1.0,
        "cer": float(cer(references, hypotheses)) if references else 1.0,
    }
    return {
        "metrics": metrics,
        "sample_predictions": sample_records,
        "total_samples": len(references),
        "evaluated_batches": batches,
    }

In [ ]:
def run_experiment(exp_key: str, max_eval_batches: Optional[int] = 5) -> Dict:
    exp = EXPERIMENTS[exp_key]
    artifact_dir = exp.artifact_dir()
    print(f"\n=== Experiment: {exp.model.name} ({exp.key}) ===")

    # Data
    raw_train, raw_test = load_afrispeech_dataset(exp.data)
    text_transform = get_text_transform(exp.data, raw_train)
    proc_train, proc_test = get_prepared_datasets(exp, text_transform, raw_train, raw_test)
    train_loader, test_loader = build_dataloaders(proc_train, proc_test, exp.train)

    # Model & optimiser
    model = build_model(exp, vocab_size=len(text_transform.token_to_id)).to(DEVICE)
    optimizer, scheduler = build_optimizer(exp, model, steps_per_epoch=len(train_loader))

    history: List[Dict[str, float]] = []
    evaluation_summaries: List[Dict[str, Any]] = []
    best_metric = math.inf
    best_snapshot: Optional[Dict[str, Any]] = None
    last_snapshot: Optional[Dict[str, Any]] = None
    best_path = artifact_dir / "best.pt"

    for epoch in range(1, exp.train.epochs + 1):
        train_loss = train_one_epoch(model, train_loader, exp, optimizer, scheduler)
        eval_summary = evaluate_model(
            model,
            test_loader,
            text_transform,
            max_batches=max_eval_batches,
            sample_limit=SAMPLE_PREVIEW_LIMIT,
        )
        metrics = eval_summary["metrics"]
        record = {
            "epoch": epoch,
            "train_loss": float(train_loss),
            "wer": float(metrics["wer"]),
            "cer": float(metrics["cer"]),
        }
        history.append(record)
        HISTORY_CACHE[exp.key] = [dict(row) for row in history]
        print(f"Epoch {epoch:02d} | loss {train_loss:.4f} | WER {metrics['wer']:.3f} | CER {metrics['cer']:.3f}")

        evaluation_record = {
            "epoch": epoch,
            "metrics": {k: float(v) for k, v in metrics.items()},
            "total_samples": int(eval_summary["total_samples"]),
            "evaluated_batches": int(eval_summary["evaluated_batches"]),
            "sample_predictions": [dict(sample) for sample in eval_summary["sample_predictions"]],
        }
        evaluation_summaries.append(evaluation_record)
        if len(evaluation_summaries) > SAMPLE_SAVE_LIMIT:
            del evaluation_summaries[0 : len(evaluation_summaries) - SAMPLE_SAVE_LIMIT]

        last_snapshot = {
            "epoch": evaluation_record["epoch"],
            "metrics": dict(evaluation_record["metrics"]),
            "sample_predictions": [dict(sample) for sample in evaluation_record["sample_predictions"]],
        }

        if metrics["wer"] < best_metric:
            best_metric = metrics["wer"]
            best_snapshot = {
                "epoch": last_snapshot["epoch"],
                "metrics": dict(last_snapshot["metrics"]),
                "sample_predictions": [dict(sample) for sample in last_snapshot["sample_predictions"]],
            }
            torch.save(model.state_dict(), best_path)

    history_path = artifact_dir / TRAINING_HISTORY_FILENAME
    with open(history_path, "w") as f:
        json.dump(history, f, indent=2)

    torch.save(model.state_dict(), artifact_dir / "last.pt")
    HISTORY_CACHE[exp.key] = [dict(row) for row in history]

    timestamp = datetime.now(timezone.utc).isoformat()

    final_metrics = {
        "experiment": exp.key,
        "model_name": exp.model.name,
        "device": DEVICE,
        "best_epoch": best_snapshot["epoch"] if best_snapshot else None,
        "best_metrics": best_snapshot["metrics"] if best_snapshot else {},
        "final_epoch": last_snapshot["epoch"] if last_snapshot else None,
        "final_metrics": last_snapshot["metrics"] if last_snapshot else {},
        "epochs_completed": len(history),
        "timestamp": timestamp,
    }
    with open(artifact_dir / FINAL_METRICS_FILENAME, "w") as f:
        json.dump(final_metrics, f, indent=2)

    sample_payload = {
        "experiment": exp.key,
        "model_name": exp.model.name,
        "device": DEVICE,
        "created_at": timestamp,
        "best_epoch": best_snapshot["epoch"] if best_snapshot else None,
        "best_samples": best_snapshot["sample_predictions"] if best_snapshot else [],
        "final_epoch": last_snapshot["epoch"] if last_snapshot else None,
        "final_samples": last_snapshot["sample_predictions"] if last_snapshot else [],
        "evaluations": evaluation_summaries,
    }
    with open(artifact_dir / SAMPLE_PREDICTIONS_FILENAME, "w") as f:
        json.dump(sample_payload, f, indent=2)

    return {
        "history": history,
        "text_transform": text_transform,
        "artifact_dir": artifact_dir,
        "model": model,
        "best_metrics": best_snapshot["metrics"] if best_snapshot else {},
        "final_metrics": last_snapshot["metrics"] if last_snapshot else {},
        "evaluation_summaries": evaluation_summaries,
        "sample_predictions_path": artifact_dir / SAMPLE_PREDICTIONS_FILENAME,
        "final_metrics_path": artifact_dir / FINAL_METRICS_FILENAME,
    }

## 8. Running a Sweep
Toggle `SELECTED_EXPERIMENTS` to run one or all backbones. Reduce epochs or `max_eval_batches` for quick smoke tests.


In [ ]:
SELECTED_EXPERIMENTS = ["vanilla_rnn"] #"gru", "lstm", "attention"
RUN_TRAINING = True  # Switch to True when you are ready to train
MAX_EVAL_BATCHES = 3  # keep small for quick validation

results = {}
if RUN_TRAINING:
    for key in SELECTED_EXPERIMENTS:
        outputs = run_experiment(key, max_eval_batches=MAX_EVAL_BATCHES)
        results[key] = outputs
else:
    print("Training skipped. Set RUN_TRAINING = True to execute the sweep.")


In [ ]:
# Inspect a few samples from the raw and processed datasets
exp = EXPERIMENTS["vanilla_rnn"] # Using vanilla_rnn config for data loading
raw_train, raw_test = load_afrispeech_dataset(exp.data)
text_transform = get_text_transform(exp.data, raw_train)
extractor = build_feature_extractor(exp.data)

print("Inspecting a few samples:")
for i in range(min(5, len(raw_test))):
    sample = raw_test[i]
    waveform = sample["audio"]["array"]
    sample_rate = sample["audio"]["sampling_rate"]
    raw_transcript = sample["transcript"]
    normalized_transcript = normalize_text(raw_transcript)
    features = extractor(waveform, sample_rate)
    labels = text_transform.text_to_ids(normalized_transcript)

    print(f"\n--- Sample {i+1} ---")
    print(f"Raw Transcript: {raw_transcript}")
    print(f"Normalized Transcript: {normalized_transcript}")
    print(f"Features shape: {features.shape}")
    print(f"Labels (IDs): {labels}")
    print(f"Labels (Text): {text_transform.ids_to_text(labels)}")

print(f"\nVocabulary size: {len(text_transform.tokens)}")
print(f"Vocabulary: {text_transform.tokens}")

Inspecting a few samples:

--- Sample 1 ---
Raw Transcript: As a result, the DNA in a double helix is arranged incomplementary strands: the sequence of nucleotides in one strand of DNA is a mirror image of the nucleotide sequence in the other DNA strand.
Normalized Transcript: as a result the dna in a double helix is arranged incomplementary strands the sequence of nucleotides in one strand of dna is a mirror image of the nucleotide sequence in the other dna strand
Features shape: torch.Size([1514, 80])
Labels (IDs): [4, 22, 1, 4, 1, 21, 8, 22, 24, 15, 23, 1, 23, 11, 8, 1, 7, 17, 4, 1, 12, 17, 1, 4, 1, 7, 18, 24, 5, 15, 8, 1, 11, 8, 15, 12, 27, 1, 12, 22, 1, 4, 21, 21, 4, 17, 10, 8, 7, 1, 12, 17, 6, 18, 16, 19, 15, 8, 16, 8, 17, 23, 4, 21, 28, 1, 22, 23, 21, 4, 17, 7, 22, 1, 23, 11, 8, 1, 22, 8, 20, 24, 8, 17, 6, 8, 1, 18, 9, 1, 17, 24, 6, 15, 8, 18, 23, 12, 7, 8, 22, 1, 12, 17, 1, 18, 17, 8, 1, 22, 23, 21, 4, 17, 7, 1, 18, 9, 1, 7, 17, 4, 1, 12, 22, 1, 4, 1, 16, 12, 21, 21, 18, 21, 1,

## 9. Visual Analytics
Interactive dashboards now render immediately when this cell runs. Loss, WER, and CER each get their own pane, plus bar charts and tables summarising best epochs. Use `render_training_dashboard(prefer_disk=True)` if you only have saved histories.


In [ ]:
def load_history(exp_key: str, prefer_disk: bool = False) -> List[Dict[str, float]]:
    if not prefer_disk:
        cached = HISTORY_CACHE.get(exp_key)
        if cached:
            return cached
        in_session = results.get(exp_key, {}).get("history") if "results" in globals() else None
        if in_session:
            history_copy = [dict(row) for row in in_session]
            HISTORY_CACHE[exp_key] = history_copy
            return history_copy
    path = EXPERIMENTS[exp_key].artifact_dir(create=False) / TRAINING_HISTORY_FILENAME
    if not path.exists():
        return []
    with open(path) as f:
        history: List[Dict[str, float]] = json.load(f)
    HISTORY_CACHE[exp_key] = history
    return history


def load_final_metrics(exp_key: str) -> Dict[str, Any]:
    path = EXPERIMENTS[exp_key].artifact_dir(create=False) / FINAL_METRICS_FILENAME
    if not path.exists():
        return {}
    with open(path) as f:
        return json.load(f)


def load_sample_predictions(exp_key: str) -> Dict[str, Any]:
    path = EXPERIMENTS[exp_key].artifact_dir(create=False) / SAMPLE_PREDICTIONS_FILENAME
    if not path.exists():
        return {}
    with open(path) as f:
        return json.load(f)


def normalize_history(history: List[Dict[str, float]]) -> List[Dict[str, float]]:
    if not history:
        return history

    normalized_history = []
    metrics = ["train_loss", "wer", "cer"]
    for metric in metrics:
        values = [row.get(metric) for row in history if row.get(metric) is not None]
        if not values:
            continue
        min_val = min(values)
        max_val = max(values)
        if max_val == min_val:
            scaled_values = [0.0] * len(values)
        else:
            scaled_values = [(v - min_val) / (max_val - min_val) for v in values]

        idx = 0
        for row in history:
            normalized_row = dict(row)
            if row.get(metric) is not None:
                normalized_row[f"{metric}_normalized"] = scaled_values[idx]
                idx += 1
            normalized_history.append(normalized_row)
    return normalized_history


def plot_metric_grid(histories: Dict[str, List[Dict[str, float]]]) -> bool:
    valid_histories = {key: hist for key, hist in histories.items() if hist}
    if not valid_histories:
        print(f"No training histories found yet. Train a model or load {TRAINING_HISTORY_FILENAME} files, then re-run this cell.")
        return False

    metrics = ["train_loss", "wer", "cer"]
    titles = ["Training Loss Over Epochs", "Word Error Rate (WER) Over Epochs", "Character Error Rate (CER) Over Epochs"]
    y_titles = ["Loss Value", "Rate", "Rate"]

    for i, metric in enumerate(metrics):
        fig = go.Figure()
        for key, history in sorted(valid_histories.items()):
            epochs = [row["epoch"] for row in history]
            fig.add_trace(
                go.Scatter(
                    x=epochs,
                    y=[row.get(metric) for row in history],
                    mode="lines+markers",
                    name=f"{key} {metric.replace('_', ' ')}",
                    line=dict(width=2),
                )
            )
        fig.update_layout(
            title=titles[i],
            xaxis_title="Epoch",
            yaxis_title=y_titles[i],
            template="plotly_white",
            hovermode="x unified",
            legend=dict(orientation="h", y=-0.2),
        )
        fig.show()

    return True


def render_metric_summary(histories: Dict[str, List[Dict[str, float]]]) -> None:
    summary_rows = []
    for key, history in histories.items():
        if not history:
            continue
        best_wer = min(history, key=lambda row: row["wer"])
        best_cer = min(history, key=lambda row: row["cer"])
        summary_rows.append(
            {
                "experiment": key,
                "best_WER": best_wer["wer"],
                "best_WER_epoch": best_wer["epoch"],
                "best_CER": best_cer["cer"],
                "best_CER_epoch": best_cer["epoch"],
            }
        )

    if not summary_rows:
        print("No summary data available yet.")
        return

    df = pd.DataFrame(summary_rows).sort_values("best_WER")
    display(df)


def render_history_table(histories: Dict[str, List[Dict[str, float]]]) -> None:
    records = []
    for key, history in histories.items():
        if not history:
            continue
        final = history[-1]
        best_wer = min(history, key=lambda row: row["wer"])
        best_cer = min(history, key=lambda row: row["cer"])
        records.append(
            {
                "experiment": key,
                "epochs": len(history),
                "final_epoch": final.get("epoch"),
                "final_loss": final.get("train_loss"),
                "final_WER": final.get("wer"),
                "final_CER": final.get("cer"),
                "best_WER": best_wer["wer"],
                "best_WER_epoch": best_wer["epoch"],
                "best_CER": best_cer["cer"],
                "best_CER_epoch": best_cer["epoch"],
            }
        )

    if not records:
        print("No summary data available yet.")
        return

    df = pd.DataFrame(records).sort_values("best_WER")
    display(df)


def render_epoch_detail_table(histories: Dict[str, List[Dict[str, float]]], max_rows: int = 80) -> None:
    long_rows = []
    for key, history in histories.items():
        for row in history:
            long_rows.append(
                {
                    "experiment": key,
                    "epoch": row.get("epoch"),
                    "train_loss": row.get("train_loss"),
                    "wer": row.get("wer"),
                    "cer": row.get("cer"),
                }
            )

    if not long_rows:
        return

    df = pd.DataFrame(long_rows).sort_values(["experiment", "epoch"])
    truncated = df.head(max_rows)
    if len(df) > max_rows:
        print(f"Showing first {max_rows} rows (of {len(df)}).")

    display(truncated)


def load_results_for_plot(keys: List[str], prefer_disk: bool = False) -> Tuple[Dict[str, List[Dict[str, float]]], List[str]]:
    histories: Dict[str, List[Dict[str, float]]] = {}
    missing: List[str] = []
    for key in keys:
        history = load_history(key, prefer_disk=prefer_disk)
        if history:
            histories[key] = history
        else:
            missing.append(key)
    return histories, missing


def render_training_dashboard(
    keys: Optional[List[str]] = None,
    prefer_disk: bool = False,
    auto: bool = False,
    max_rows: int = 80,
    ) -> None:
    keys = keys or SELECTED_EXPERIMENTS
    histories, missing = load_results_for_plot(keys, prefer_disk=prefer_disk)
    if not histories:
        if not auto:
            print(f"No training runs available. Train models or provide {TRAINING_HISTORY_FILENAME} files, then rerun `render_training_dashboard`.")
        return
    if missing and not auto:
        print(f"No history yet for: {', '.join(missing)}")
    if not plot_metric_grid(histories):
        return
    render_metric_summary(histories)
    render_history_table(histories)
    render_epoch_detail_table(histories, max_rows=max_rows)


render_training_dashboard(auto=True)

### Attention Heatmaps
When an attention model is trained, capture its latest attention map and render it as a Plotly heatmap for qualitative analysis.


In [ ]:
def plot_attention_heatmap(model: PrositAcousticModel, title: str = "Attention Map") -> None:
    attn = model.attention_map()
    if attn is None:
        print("No attention weights captured. Train an attention model and run a forward pass first.")
        return
    array = attn.cpu().numpy()
    fig = go.Figure(data=go.Heatmap(z=array, colorscale="Viridis"))
    fig.update_layout(title=title, xaxis_title="Source frames", yaxis_title="Target frames")
    fig.show()


def demo_attention_forward(exp_key: str) -> None:
    if exp_key not in results:
        print("Attention model not in `results`. Either run training in this session or load the model manually.")
        return
    record = results[exp_key]
    model = record["model"]
    text_transform = record["text_transform"]

    raw_train, raw_test = load_afrispeech_dataset(EXPERIMENTS[exp_key].data)
    extractor = build_feature_extractor(EXPERIMENTS[exp_key].data)
    sample = raw_test[0]
    features = extractor(sample["audio"]["array"], sample["audio"]["sampling_rate"]).unsqueeze(1).to(DEVICE)
    log_probs = model(features)
    model.greedy_decode(log_probs.cpu(), torch.tensor([features.shape[0]]), blank_id=0, text_transform=text_transform)
    plot_attention_heatmap(model, title=f"Attention ({EXPERIMENTS[exp_key].model.name})")


## 10. Inference Utility
Load a waveform, reuse the trained text transform, and decode with greedy CTC for quick demos or qualitative checks.


In [ ]:
@torch.no_grad()
def transcribe_waveform(model: PrositAcousticModel, waveform: np.ndarray, sample_rate: int, data_cfg: DataConfig, text_transform: TextTransform) -> str:
    extractor = build_feature_extractor(data_cfg)
    features = extractor(waveform, sample_rate)
    tensor = features.unsqueeze(1).to(DEVICE)
    log_probs = model(tensor)
    transcript = model.greedy_decode(log_probs.cpu(), torch.tensor([features.shape[0]]), blank_id=0, text_transform=text_transform)[0]
    return transcript


### Reference vs Prediction Table
Use the helper below after training to inspect decoded outputs alongside ground truth and per-sample WER/CER.


In [ ]:
@torch.no_grad()
def preview_predictions(exp_key: str, num_samples: int = 5, split: str = "test") -> None:
    if exp_key not in results:
        print("Experiment not found in `results`. Run training first in this session.")
        return

    if split not in {"train", "test"}:
        raise ValueError("split must be 'train' or 'test'")

    record = results[exp_key]
    model = record["model"]
    text_transform = record["text_transform"]
    exp = EXPERIMENTS[exp_key]

    raw_train, raw_test = load_afrispeech_dataset(exp.data)
    extractor = build_feature_extractor(exp.data)
    dataset = raw_test if split == "test" else raw_train
    processed = prepare_dataset(dataset, extractor, text_transform)

    rows = []
    total = min(num_samples, len(processed))
    for idx in range(total):
        item = processed[idx]
        features = item["input_features"]
        if not isinstance(features, torch.Tensor):
            features = torch.tensor(features, dtype=torch.float32)
        features = features.unsqueeze(1).to(DEVICE)
        log_probs = model(features)
        prediction = model.greedy_decode(
            log_probs.cpu(),
            torch.tensor([features.shape[0]]),
            blank_id=0,
            text_transform=text_transform,
        )[0]
        reference = item["normalized_transcript"]
        rows.append(
            {
                "sample_index": idx,
                "reference": reference,
                "prediction": prediction,
                "sample_WER": wer([reference], [prediction]),
                "sample_CER": cer([reference], [prediction]),
            }
        )

    if not rows:
        print("No samples available for preview.")
        return

    df = pd.DataFrame(rows)
    table = go.Figure(
        data=[
            go.Table(
                header=dict(values=df.columns, fill_color="#222", font_color="white", align="left"),
                cells=dict(values=[df[col] for col in df.columns], align="left"),
            )
        ]
    )
    table.update_layout(
        title=f"Reference vs Prediction — {exp.model.name} ({split} split)",
        template="plotly_white",
    )
    table.show()


### Next Steps
- Increase `DataConfig.fraction` and training epochs for full-scale runs.
- Swap `SELECTED_EXPERIMENTS` to focus on a single backbone when iterating quickly.
- Use `demo_attention_forward("attention")` after training to inspect where the model focuses.
